In [1]:
pip install yt-dlp

   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ---------------- ----------------------- 1.3/3.2 MB 9.6 MB/s eta 0:00:01
   ---------------------------------------- 3.2/3.2 MB 9.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [5]:
pip install pandas requests yt-dlp openpyxl bs4

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import csv
import json
import time
import random
import warnings
import requests
import pandas as pd

warnings.filterwarnings("ignore")

# =========================================================
# KONFIGURASI LOKASI FILE
# =========================================================

BASE_DIR = r"D:\New Folder (3)"
INPUT_FILE = os.path.join(BASE_DIR, "data_tiktok.xlsx")
OUTPUT_FILE = os.path.join(BASE_DIR, "hasil_tiktok_final.csv")

TIMEOUT = 15
SLEEP_MIN = 0.5
SLEEP_MAX = 1.2

# Gunakan Mobile User-Agent agar tidak terkena SlardarWAF TikTok Desktop
# dan agar JSON __UNIVERSAL_DATA_FOR_REHYDRATION__ selalu disertakan untuk /video/ maupun /photo/
MOBILE_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Linux; Android 10; K) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/130.0.0.0 Mobile Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
}

DESKTOP_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

session = requests.Session()
session.headers.update(MOBILE_HEADERS)


def safe_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip()


def extract_username_from_url(url):
    match = re.search(r"/@([^/?#]+)", safe_text(url))
    if match:
        return match.group(1).strip().lstrip("@")
    return ""


def extract_item_id_from_url(url):
    match = re.search(r"/(?:video|photo)/(\d+)", safe_text(url))
    if match:
        return match.group(1)
    return ""


def get_tiktok_stats(url, raw_username):
    """
    Menarik metrik engagement TikTok (mendukung /video/ maupun /photo/ slideshow)
    beserta deteksi Keranjang Kuning (TikTok Shop) melalui anchors.
    """
    url = safe_text(url)
    username = safe_text(raw_username).lstrip("@")
    if username.lower() == "nan":
        username = ""

    if not url:
        return username, "URL Kosong", "", "", "", "", "", "", ""

    target_url = url
    if "tiktok.com/@" in target_url and "?" not in target_url:
        target_url = target_url + "?_r=1"

    real_url = target_url
    status_code = None

    # ---------------------------------------------------------
    # LAPIS 1: TikTok Mobile Reflow HTML Parser (dengan retry)
    # ---------------------------------------------------------
    for attempt in range(3):
        try:
            res = session.get(target_url, allow_redirects=True, timeout=TIMEOUT)
            real_url = res.url or target_url

            if not username:
                username = extract_username_from_url(real_url)

            if "tiktok.com/@" in real_url and "?" not in real_url:
                res = session.get(real_url + "?_r=1", allow_redirects=True, timeout=TIMEOUT)

            match = re.search(
                r'<script id="__UNIVERSAL_DATA_FOR_REHYDRATION__"[^>]*>(.*?)</script>',
                res.text,
            )
            if match:
                data = json.loads(match.group(1))
                scope = data.get("__DEFAULT_SCOPE__", {})
                detail = (
                    scope.get("webapp.reflow.video.detail")
                    or scope.get("webapp.video-detail")
                    or {}
                )
                status_code = detail.get("statusCode")

                item_struct = detail.get("itemInfo", {}).get("itemStruct", {})
                stats = item_struct.get("stats")
                author = item_struct.get("author", {})

                if author and author.get("uniqueId") and not username:
                    username = author.get("uniqueId")

                if stats and status_code == 0:
                    views = stats.get("playCount", 0)
                    likes = stats.get("diggCount", 0)
                    comments = stats.get("commentCount", 0)
                    shares = stats.get("shareCount", 0)
                    saved = stats.get("collectCount", 0)
                    
                    # --- DETEKSI KERANJANG KUNING ---
                    keranjang = ""
                    anchors = item_struct.get("anchors", [])
                    if isinstance(anchors, list):
                        for anc in anchors:
                            anc_str = str(anc).lower()
                            if anc.get("type") == 3 or "shop" in anc_str or "product" in anc_str or "commodity" in anc_str:
                                keranjang = "V"
                                break
                    # --------------------------------

                    return username or "Gagal_Ekstrak", views, likes, comments, shares, saved, "", "", keranjang

                if status_code == 10204:
                    break
        except Exception:
            time.sleep(1.0)

    # ---------------------------------------------------------
    # LAPIS 2: TikTok Embed v2 Endpoint Fallback
    # ---------------------------------------------------------
    item_id = extract_item_id_from_url(real_url)
    if item_id and status_code != 10204:
        try:
            embed_url = f"https://www.tiktok.com/embed/v2/{item_id}"
            res_emb = requests.get(embed_url, headers=DESKTOP_HEADERS, timeout=TIMEOUT)
            m_emb = re.search(
                r'<script id="__FRONTITY_CONNECT_STATE__"[^>]*>(.*?)</script>',
                res_emb.text,
            )
            if m_emb:
                emb_data = json.loads(m_emb.group(1))
                vdata = (
                    emb_data.get("source", {})
                    .get("data", {})
                    .get(f"/embed/v2/{item_id}", {})
                    .get("videoData", {})
                )
                item_infos = vdata.get("itemInfos", {})
                author_infos = vdata.get("authorInfos", {})
                if author_infos.get("uniqueId") and not username:
                    username = author_infos.get("uniqueId")
                if item_infos:
                    views = item_infos.get("playCount", 0)
                    likes = item_infos.get("diggCount", 0)
                    comments = item_infos.get("commentCount", 0)
                    shares = item_infos.get("shareCount", 0)
                    saved = item_infos.get("collectCount", "")
                    return username or "Gagal_Ekstrak", views, likes, comments, shares, saved, "", "", ""
        except Exception:
            pass

    # ---------------------------------------------------------
    # LAPIS 3: TikWM API Fallback
    # ---------------------------------------------------------
    if status_code != 10204:
        try:
            clean_url = real_url.split("?")[0]
            res_wm = requests.post(
                "https://www.tikwm.com/api/",
                data={"url": clean_url},
                timeout=TIMEOUT,
            ).json()
            if res_wm.get("code") == 0 and res_wm.get("data"):
                d = res_wm["data"]
                if not username:
                    username = (d.get("author") or {}).get("unique_id", "")
                views = d.get("play_count", 0)
                likes = d.get("digg_count", 0)
                comments = d.get("comment_count", 0)
                shares = d.get("share_count", 0)
                saved = d.get("collect_count", 0)
                return username or "Gagal_Ekstrak", views, likes, comments, shares, saved, "", "", ""
        except Exception:
            pass

    if not username:
        username = extract_username_from_url(real_url) or "Gagal_Ekstrak"

    if status_code == 10204:
        err_msg = "Postingan Tidak Tersedia / Akun Banned (10204)"
    else:
        err_msg = "Gagal Ekstrak"

    return username, err_msg, "", "", "", "", "", "", ""


# =========================================================
# PROGRAM UTAMA
# =========================================================

def main():
    if not os.path.exists(INPUT_FILE):
        raise FileNotFoundError(f"File tidak ditemukan: {INPUT_FILE}")

    df = pd.read_excel(INPUT_FILE)
    if "Link" not in df.columns:
        raise ValueError("Kolom 'Link' wajib ada di file Excel.")

    # Menambahkan kolom "Keranjang Kuning" di CSV Output
    output_columns = [
        "Date", "Kreator", "Username", "Link",
        "Views", "Likes", "Comments",
        "Shares", "Saved", "Top Comment Text", "Top Comment Likes",
        "Keranjang Kuning"
    ]

    total = len(df)
    print(f"Input      : {INPUT_FILE}")
    print(f"Total data : {total}")
    print(f"Output     : {OUTPUT_FILE}\n")

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=output_columns)
        writer.writeheader()
        f.flush()

        for i, (_, row) in enumerate(df.iterrows(), start=1):
            url = safe_text(row.get("Link", ""))
            date_val = safe_text(row.get("Date", ""))
            creator = safe_text(row.get("Kreator", ""))
            raw_username = row.get("Username", "")

            print(f"[{i}/{total}] Memproses: {url}")

            if not url:
                username = safe_text(raw_username)
                views, likes, comments, shares, saved, top_text, top_likes, keranjang = (
                    "Link Kosong", "", "", "", "", "", "", ""
                )
            else:
                # Menerima 9 variabel balikan dari fungsi
                username, views, likes, comments, shares, saved, top_text, top_likes, keranjang = (
                    get_tiktok_stats(url, raw_username)
                )

            writer.writerow({
                "Date": date_val,
                "Kreator": creator,
                "Username": username,
                "Link": url,
                "Views": views,
                "Likes": likes,
                "Comments": comments,
                "Shares": shares,
                "Saved": saved,
                "Top Comment Text": top_text,
                "Top Comment Likes": top_likes,
                "Keranjang Kuning": keranjang,
            })

            f.flush()
            
            # Print informasi log di terminal
            keranjang_info = "[KERANJANG]" if keranjang == "V" else ""
            print(
                f"   -> Username: {username} | Views: {views} | Likes: {likes} "
                f"| Comments: {comments} {keranjang_info}"
            )

            if i < total:
                time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))

    print("\nProses Selesai!")
    print(f"Hasil tersimpan di: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Input      : D:\New Folder (3)\data_tiktok.xlsx
Total data : 1304
Output     : D:\New Folder (3)\hasil_tiktok_final.csv

[1/1304] Memproses: https://vt.tiktok.com/ZSbtFMM99/
   -> Username: menaana_ | Views: 176 | Likes: 15 | Comments: 0 
[2/1304] Memproses: https://vt.tiktok.com/ZSbtFMRD8/
   -> Username: menaana_ | Views: 397 | Likes: 23 | Comments: 2 
[3/1304] Memproses: https://vt.tiktok.com/ZSbtFLXn6/
   -> Username: menaana_ | Views: 217 | Likes: 10 | Comments: 0 
[4/1304] Memproses: https://vt.tiktok.com/ZSbtFhMDe/
   -> Username: menaana_ | Views: 264 | Likes: 10 | Comments: 0 
[5/1304] Memproses: https://vt.tiktok.com/ZSbtFqqns/
   -> Username: menaana_ | Views: 232 | Likes: 10 | Comments: 0 [KERANJANG]
[6/1304] Memproses: https://vt.tiktok.com/ZSbtFyo3b/
   -> Username: menaana_ | Views: 150 | Likes: 10 | Comments: 0 [KERANJANG]
[7/1304] Memproses: https://vt.tiktok.com/ZSbtF78rk/
   -> Username: menaana_ | Views: 158 | Likes: 10 | Comments: 0 
[8/1304] Memproses: https://vt.